<center><font color='steelblue'> <font size = "6">CORPORACIÓN UNIVERSITARIA MINUTO DE DIOS </font></center><br>
<center><font color='steelblue'> <font size = "5,5">ESCUELA DE INGENIERÍA</font></center><br>
<center><font color='steelblue'> <font size = "5">PROGRAMA INGENIERÍA DE SISTEMAS - Modalidad Presencial</font></center><br>
<center><font color='steelblue'> <font size = "4">BASES DE DATOS MASIVAS</font></center><br>
<center><font color="yellow" size = "4" face = "small fonts">Proyecto Segundo Corte – Arquitectura distribuida, ETL y Mojo</font></center>

**Modalidad:** Aprendizaje Basado en Proyectos (ABP)  
**Entregable:** Un único archivo `ApellidoNombre_ProyectoBDM.ipynb`

> **Propósito:** construir y demostrar una solución distribuida que integre MongoDB, Docker, Mojo (Modular) y una base de datos relacional mediante un proceso ETL reproducible.

Este cuaderno es la guía oficial de desarrollo. El equipo debe completar las celdas marcadas como `TODO`, documentar las decisiones técnicas y conservar las evidencias de ejecución.

## 0. Datos del equipo

Complete esta información antes de comenzar.

- **NRC:** 83355
- **Asignatura:** Bases de Datos Masivas
- **Docente:** Alonso Guevara Perez
- **Integrantes:**  Valery Pamplona, Paola Cerpa , Brahiam Yagama 
- **Fecha:** 15 de octubre de 2026
- **Sistema operativo:** Windows + WSL2 (Ubuntu)
- **Versión Docker:** 29.4.3 (Docker Compose v5.1.3)
- **Versión Mojo:** 1.1.0 (8189361e)
- **Versión Python:** 3.12.3 (Ubuntu en WSL2)

### Integridad académica

El código debe poder ser explicado por todos los integrantes. La inteligencia artificial puede utilizarse como apoyo para comprender, depurar o consultar documentación, pero no sustituye la comprensión.

**Regla:** aportes generados exclusivamente por IA, sin comprensión, adaptación, validación y explicación del estudiante, podrán recibir **0.0** en el componente correspondiente.

# 1. Contexto y reto

La organización dispone de tres fuentes JSON:

- `personas.json`
- `articulos.json`
- `ventas.json`

La información deberá distribuirse en tres nodos MongoDB independientes, procesarse mediante Mojo y finalmente integrarse en un modelo relacional.

### Reto

> ¿Cómo diseñar e implementar una arquitectura distribuida que permita extraer, validar, transformar e integrar datos heterogéneos, garantizando integridad referencial y produciendo información útil para el análisis?

### Condición fundamental

El lenguaje utilizado para la **lógica central de limpieza, validación y transformación** debe ser **Mojo de Modular**.

**No se acepta Mojolicious/Perl como sustituto de Mojo.**

Python puede utilizarse para orquestación desde Jupyter, automatización, visualización y apoyo al proceso, siempre que el equipo diferencie claramente qué responsabilidad corresponde a Python y cuál a Mojo.

# 2. Arquitectura objetivo

```text
                    ┌─────────────────────┐
                    │   ARCHIVOS JSON     │
                    │ personas/articulos  │
                    │ ventas              │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │   RED DOCKER        │
                    └──────────┬──────────┘
                               │
            ┌──────────────────┼──────────────────┐
            │                  │                  │
            ▼                  ▼                  ▼
      ┌───────────┐      ┌───────────┐      ┌───────────┐
      │ Mongo 1   │      │ Mongo 2   │      │ Mongo 3   │
      │ PERSONAS  │      │ ARTICULOS │      │ VENTAS    │
      └─────┬─────┘      └─────┬─────┘      └─────┬─────┘
            │                  │                  │
            └──────────────────┼──────────────────┘
                               ▼
                    ┌─────────────────────┐
                    │       MOJO          │
                    │ Extract / Validate  │
                    │ Transform / Integrate│
                    └──────────┬──────────┘
                               ▼
                    ┌─────────────────────┐
                    │ BD RELACIONAL       │
                    │ PERSONAS/ARTICULOS  │
                    │ VENTAS              │
                    └──────────┬──────────┘
                               ▼
                    ┌─────────────────────┐
                    │ SQL + VISUALIZACIÓN │
                    └─────────────────────┘
```

### Actividad

Explique:

1. ¿Por qué existen tres nodos MongoDB?
2. ¿Qué ventaja ofrece Docker?
3. ¿Qué responsabilidad tiene Mojo?
4. ¿Por qué se necesita un modelo relacional al final?
5. ¿Dónde se garantiza la integridad referencial?

El equipo debe reproducir o mejorar este diagrama.

# 3. Modelo relacional de referencia

El equipo debe construir el DDL dentro del notebook. **No se entrega una base SQLite ya poblada.**

# 4. Preparación del entorno

El notebook debe crear una estructura reproducible:

```text
proyecto/
├── data/
│   ├── personas.json
│   ├── articulos.json
│   └── ventas.json
├── mojo/
│   ├── src/
│   └── tests/
├── docker/
├── sql/
└── resultados/
```

No utilice rutas absolutas del equipo. Use rutas relativas o dinámicas.

In [1]:
from pathlib import Path
import os
import sys
import json
import re
import sqlite3
import subprocess
import platform

BASE = Path.cwd()
DATA = BASE / "data"
MOJO_DIR = BASE / "mojo"
DOCKER_DIR = BASE / "docker"
SQL_DIR = BASE / "sql"
RESULTADOS = BASE / "resultados"

for p in [DATA, MOJO_DIR, DOCKER_DIR, SQL_DIR, RESULTADOS]:
    p.mkdir(parents=True, exist_ok=True)

print("Directorio:", BASE)
print("Sistema operativo:", platform.platform())
print("Python:", sys.version.split()[0])

Directorio: /root/proyectoBDM
Sistema operativo: Linux-6.18.40.1-microsoft-standard-WSL2-x86_64-with-glibc2.39
Python: 3.12.3


## 4.1 Verificación de archivos

Los tres JSON deben estar disponibles en `data/`. Si el notebook se encuentra en otro directorio, adapte la ruta de manera dinámica.

In [2]:
required = ["personas.json", "articulos.json", "ventas.json"]

for name in required:
    path = DATA / name
    print(f"{name:20} ->", "OK" if path.exists() else "FALTA", path)

personas.json        -> OK /root/proyectoBDM/data/personas.json
articulos.json       -> OK /root/proyectoBDM/data/articulos.json
ventas.json          -> OK /root/proyectoBDM/data/ventas.json


# 5. Perfilamiento inicial de los datos

Antes de transformar información, determine qué contiene realmente cada fuente.

Para cada JSON calcule:

- Número de registros.
- Campos disponibles.
- Campos faltantes.
- Tipos observados.
- Valores potencialmente inválidos.
- Duplicados de las claves.
- Ejemplos de registros problemáticos.

**No modifique los archivos originales.**

In [3]:
def cargar_json(nombre):
    path = DATA / nombre
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

personas = cargar_json("personas.json")
articulos = cargar_json("articulos.json")
ventas = cargar_json("ventas.json")

print("Personas :", len(personas))
print("Articulos:", len(articulos))
print("Ventas   :", len(ventas))

Personas : 10000
Articulos: 10000
Ventas   : 10000


In [4]:
from collections import Counter
from datetime import datetime
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

# Decisión técnica: el perfilador SOLO lee y reporta; no modifica ni corrige nada.
# Cada campo tiene una regla de validez; lo que no la cumple se cuenta como "inválido".

def _es_fecha(v):
    try:
        datetime.strptime(str(v), "%Y-%m-%d")
        return True
    except ValueError:
        return False

def _solo_digitos(v):
    return isinstance(v, (str, int)) and re.fullmatch(r"\d+", str(v).strip()) is not None

def _texto(v):
    return isinstance(v, str) and v.strip() != ""

def _email(v):
    return isinstance(v, str) and re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", v.strip()) is not None

def _num_pos(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool) and v > 0

ESQUEMAS = {
    "PERSONAS": {
        "clave": "numeroDocumento",
        "campos": {
            "numeroDocumento": (_solo_digitos, "solo dígitos"),
            "nombres": (_texto, "texto no vacío"),
            "primerApellido": (_texto, "texto no vacío"),
            "segundoApellido": (_texto, "texto no vacío"),
            "fechaNacimiento": (_es_fecha, "fecha AAAA-MM-DD válida"),
            "telefono": (_solo_digitos, "solo dígitos"),
            "direccion": (_texto, "texto no vacío"),
            "email": (_email, "formato de correo"),
        },
    },
    "ARTICULOS": {
        "clave": "idArticulo",
        "campos": {
            "idArticulo": (_solo_digitos, "solo dígitos"),
            "nombreArticulo": (_texto, "texto no vacío"),
            "precioArticulo": (_num_pos, "número > 0"),
            "cantidadArticulo": (_solo_digitos, "solo dígitos"),
        },
    },
    "VENTAS": {
        "clave": "idVenta",
        "campos": {
            "idVenta": (_solo_digitos, "solo dígitos"),
            "idComprador": (_solo_digitos, "solo dígitos"),
            "idArticulo": (_solo_digitos, "solo dígitos"),
            "cantidadProductos": (_solo_digitos, "solo dígitos"),
            "precioTotal": (_num_pos, "número > 0"),
        },
    },
}

def perfilar_registros(registros, nombre, max_ejemplos=3):
    esquema = ESQUEMAS[nombre]
    campos_esp = esquema["campos"]
    clave = esquema["clave"]
    total = len(registros)
    print(f"\n--- {nombre} ---")

    # 1) Campos disponibles y faltantes
    presentes = Counter(k for r in registros for k in r)
    extra = [k for k in presentes if k not in campos_esp]
    print("Campos disponibles:", list(presentes))
    print("Campos faltantes:",
          {k: total - presentes.get(k, 0) for k in campos_esp if presentes.get(k, 0) < total} or "ninguno")
    if extra:
        print("Campos no esperados:", extra)

    # 2) Tipos observados, nulos e inválidos por campo
    filas, problematicos = [], {}
    for campo, (valida, regla) in campos_esp.items():
        valores = [r.get(campo) for r in registros]
        nulos = sum(v is None or (isinstance(v, str) and v.strip() == "") for v in valores)
        malos = [(i, v) for i, v in enumerate(valores) if not valida(v)]
        problematicos[campo] = malos
        filas.append({
            "campo": campo,
            "tipos": dict(Counter(type(v).__name__ for v in valores)),
            "nulos/vacíos": nulos,
            "regla": regla,
            "inválidos": len(malos),
        })
    display(pd.DataFrame(filas).set_index("campo"))

    # 3) Duplicados de la clave
    cont = Counter(r.get(clave) for r in registros)
    dup = {k: c for k, c in cont.items() if c > 1}
    print(f"Clave '{clave}': {len(cont)} valores distintos, {len(dup)} valores repetidos, "
          f"{sum(c - 1 for c in dup.values())} registros sobrantes por duplicado")
    if dup:
        print("  Ejemplos de claves duplicadas:", list(dup.items())[:max_ejemplos])

    # 4) Ejemplos de valores y registros problemáticos
    for campo, malos in problematicos.items():
        if malos:
            print(f"  Ejemplos inválidos en '{campo}':", [v for _, v in malos[:max_ejemplos]])
    indices = sorted({i for malos in problematicos.values() for i, _ in malos})
    print(f"Registros con al menos un campo inválido: {len(indices)}")
    if indices:
        print("Ejemplo de registro completo problemático:")
        print(json.dumps(registros[indices[0]], ensure_ascii=False, indent=2))

perfilar_registros(personas, "PERSONAS")
perfilar_registros(articulos, "ARTICULOS")
perfilar_registros(ventas, "VENTAS")


--- PERSONAS ---
Campos disponibles: ['numeroDocumento', 'nombres', 'primerApellido', 'segundoApellido', 'fechaNacimiento', 'telefono', 'direccion', 'email']
Campos faltantes: ninguno


,tipos,nulos/vacíos,regla,inválidos
campo,,,,
numeroDocumento,{'str': 10000},0,solo dígitos,4986
nombres,{'str': 10000},0,texto no vacío,0
primerApellido,{'str': 10000},0,texto no vacío,0
segundoApellido,{'str': 10000},0,texto no vacío,0
fechaNacimiento,{'str': 10000},0,fecha AAAA-MM-DD válida,0
telefono,{'str': 10000},0,solo dígitos,4969
direccion,{'str': 10000},0,texto no vacío,0
email,{'str': 10000},0,formato de correo,0


Clave 'numeroDocumento': 10000 valores distintos, 0 valores repetidos, 0 registros sobrantes por duplicado
  Ejemplos inválidos en 'numeroDocumento': ['@10@93@!65', '67q0$A4A1q', '!5A0q@9$8$']
  Ejemplos inválidos en 'telefono': ['422$9Aq9@1', '2@AA@2!18A', '$q$031944q']
Registros con al menos un campo inválido: 7481
Ejemplo de registro completo problemático:
{
  "numeroDocumento": "@10@93@!65",
  "nombres": "Samuel",
  "primerApellido": "Roberts",
  "segundoApellido": "Sweeney",
  "fechaNacimiento": "1938-05-30",
  "telefono": "7546027921",
  "direccion": "8818 Evans Cove Apt. 968, North Hannahville, WY 96152",
  "email": "bondanthony@gmail.com"
}

--- ARTICULOS ---
Campos disponibles: ['idArticulo', 'nombreArticulo', 'precioArticulo', 'cantidadArticulo']
Campos faltantes: ninguno


,tipos,nulos/vacíos,regla,inválidos
campo,,,,
idArticulo,{'str': 10000},0,solo dígitos,4759
nombreArticulo,{'str': 10000},0,texto no vacío,0
precioArticulo,{'float': 10000},0,número > 0,0
cantidadArticulo,{'str': 10000},0,solo dígitos,4703


Clave 'idArticulo': 6013 valores distintos, 978 valores repetidos, 3987 registros sobrantes por duplicado
  Ejemplos de claves duplicadas: [('358', 7), ('267', 8), ('791', 8)]
  Ejemplos inválidos en 'idArticulo': ['T474', '1o834', 'cO50']
  Ejemplos inválidos en 'cantidadArticulo': ['m18', '9rJ417', 'SBB462']
Registros con al menos un campo inválido: 7248
Ejemplo de registro completo problemático:
{
  "idArticulo": "T474",
  "nombreArticulo": "SeaSonic Focus Plus",
  "precioArticulo": 237.64,
  "cantidadArticulo": "m18"
}

--- VENTAS ---
Campos disponibles: ['idVenta', 'idComprador', 'idArticulo', 'cantidadProductos', 'precioTotal']
Campos faltantes: ninguno


,tipos,nulos/vacíos,regla,inválidos
campo,,,,
idVenta,{'int': 10000},0,solo dígitos,0
idComprador,{'str': 10000},0,solo dígitos,5039
idArticulo,{'int': 10000},0,solo dígitos,0
cantidadProductos,{'str': 10000},0,solo dígitos,4969
precioTotal,{'float': 10000},0,número > 0,0


Clave 'idVenta': 10000 valores distintos, 0 valores repetidos, 0 registros sobrantes por duplicado
  Ejemplos inválidos en 'idComprador': ['q61@A!213A', '!@1$@8@1@8', 'q@9042@952']
  Ejemplos inválidos en 'cantidadProductos': ['2x', '9O', '10K']
Registros con al menos un campo inválido: 7487
Ejemplo de registro completo problemático:
{
  "idVenta": 1,
  "idComprador": "q61@A!213A",
  "idArticulo": 2358,
  "cantidadProductos": "2x",
  "precioTotal": 202.14
}


**Decisiones técnicas del perfilamiento**

- El perfilador es de solo lectura: no modifica los JSON originales, que quedan en `data/`.
- Un valor es "inválido" cuando no cumple la regla de su campo (por ejemplo, un documento o teléfono con caracteres que no son dígitos).
- Se reporta también la clave de cada fuente (`numeroDocumento`, `idArticulo`, `idVenta`) para detectar duplicados antes de la carga.
- El perfilamiento solo detecta y cuantifica; la corrección o descarte se decide en la sección de calidad de datos.

**Hallazgos del perfilamiento**

- **Personas:** ningún campo faltante ni nulo. `numeroDocumento` (4.986) y `telefono` (4.969) traen caracteres que no son dígitos (por ejemplo `@10@93@!65`). La clave no tiene duplicados.
- **Artículos:** ningún campo faltante ni nulo. `idArticulo` (4.759) y `cantidadArticulo` (4.703) traen letras o símbolos mezclados. La clave `idArticulo` solo tiene 6.013 valores distintos en 10.000 registros (3.987 sobrantes por duplicado).
- **Ventas:** ningún campo faltante ni nulo. `idComprador` (5.039) y `cantidadProductos` (4.969) traen caracteres no numéricos (por ejemplo `2x`, `10K`). `idVenta` no tiene duplicados.
- **Tipos heterogéneos:** `idArticulo` es texto en artículos y entero en ventas, por lo que habrá que unificar el tipo antes de relacionar ambas tablas.
- **Pendiente para la sección de calidad:** decidir, campo por campo, si el valor inválido se recupera (por ejemplo, quitando los caracteres no numéricos) o se descarta.

## 5.1 Análisis de calidad

Investigue especialmente:

### PERSONAS
- `numeroDocumento`
- `telefono`

### ARTICULOS
- `idArticulo`
- `cantidadArticulo`

### VENTAS
- `idComprador`
- `idArticulo`
- `cantidadProductos`
- `precioTotal`

Defina una regla de negocio y justifíquela para cada tipo de inconsistencia.

**No elimine registros sin explicar la decisión.**

In [5]:
NUMERIC_RE = re.compile(r"^\d+(?:\.\d+)?$")

def es_numerico(valor):
    # Decisión: se evalúa el texto del valor; None y booleanos no son numéricos.
    if valor is None or isinstance(valor, bool):
        return False
    return NUMERIC_RE.match(str(valor).strip()) is not None

def contar_invalidos(registros, campo):
    # Un campo ausente (None) cuenta como inválido.
    return sum(1 for r in registros if not es_numerico(r.get(campo)))

print("personas.numeroDocumento :", contar_invalidos(personas, "numeroDocumento"))
print("personas.telefono        :", contar_invalidos(personas, "telefono"))
print("articulos.idArticulo     :", contar_invalidos(articulos, "idArticulo"))
print("articulos.cantidadArticulo:", contar_invalidos(articulos, "cantidadArticulo"))
print("ventas.idComprador       :", contar_invalidos(ventas, "idComprador"))
print("ventas.idArticulo        :", contar_invalidos(ventas, "idArticulo"))
print("ventas.cantidadProductos :", contar_invalidos(ventas, "cantidadProductos"))
print("ventas.precioTotal       :", contar_invalidos(ventas, "precioTotal"))

personas.numeroDocumento : 4986
personas.telefono        : 4969
articulos.idArticulo     : 4759
articulos.cantidadArticulo: 4703
ventas.idComprador       : 5039
ventas.idArticulo        : 0
ventas.cantidadProductos : 4969
ventas.precioTotal       : 0


In [6]:
def solo_digitos(v):
    return re.sub(r"\D", "", str(v))

def dist_longitudes(valores):
    return dict(sorted(Counter(len(v) for v in valores).items()))

def perfil_longitudes(registros, campo):
    validos = [str(r[campo]) for r in registros if es_numerico(r[campo])]
    malos = [str(r[campo]) for r in registros if not es_numerico(r[campo])]
    sin_digitos = sum(solo_digitos(v) == "" for v in malos)
    print(f"\n{campo}: {len(malos)} inválidos | sin ningún dígito: {sin_digitos}")
    print("  longitud de válidos              :", dist_longitudes(validos))
    print("  inválidos tras quitar no-dígitos :",
          dist_longitudes([solo_digitos(v) for v in malos if solo_digitos(v)]))

for reg, campo in [(personas, "numeroDocumento"), (personas, "telefono"),
                   (articulos, "idArticulo"), (articulos, "cantidadArticulo"),
                   (ventas, "idComprador"), (ventas, "cantidadProductos")]:
    perfil_longitudes(reg, campo)

rec = Counter(int(solo_digitos(r["cantidadProductos"])) for r in ventas if not es_numerico(r["cantidadProductos"]))
val = Counter(int(r["cantidadProductos"]) for r in ventas if es_numerico(r["cantidadProductos"]))
print("\ncantidadProductos recuperada:", sorted(rec.items()))
print("cantidadProductos válida    :", sorted(val.items()))
pt = [r["precioTotal"] for r in ventas]
print("precioTotal min/max:", min(pt), max(pt))


numeroDocumento: 4986 inválidos | sin ningún dígito: 8
  longitud de válidos              : {7: 3, 8: 57, 9: 447, 10: 4507}
  inválidos tras quitar no-dígitos : {1: 51, 2: 225, 3: 623, 4: 1080, 5: 1194, 6: 986, 7: 571, 8: 207, 9: 41}

telefono: 4969 inválidos | sin ningún dígito: 8
  longitud de válidos              : {6: 1, 7: 2, 8: 38, 9: 445, 10: 4545}
  inválidos tras quitar no-dígitos : {1: 56, 2: 252, 3: 619, 4: 1043, 5: 1205, 6: 1006, 7: 533, 8: 210, 9: 37}

idArticulo: 4759 inválidos | sin ningún dígito: 0
  longitud de válidos              : {1: 54, 2: 474, 3: 4435, 4: 229, 5: 42, 6: 7}
  inválidos tras quitar no-dígitos : {1: 46, 2: 320, 3: 3334, 4: 954, 5: 105}

cantidadArticulo: 4703 inválidos | sin ningún dígito: 0
  longitud de válidos              : {1: 482, 2: 4443, 3: 76, 4: 250, 5: 40, 6: 6}
  inválidos tras quitar no-dígitos : {1: 22, 2: 332, 3: 3314, 4: 935, 5: 100}

idComprador: 5039 inválidos | sin ningún dígito: 10
  longitud de válidos              : {7: 1, 8: 

**Interpretación de la evidencia de calidad**

- **numeroDocumento, telefono e idComprador:** los valores válidos tienen casi siempre 10 dígitos, pero los dañados, después de quitar los símbolos, conservan entre 1 y 9 (la mayoría 4 o 5). Los símbolos reemplazaron dígitos que ya no existen, así que limpiar produciría un número distinto del original. Por eso estos campos no se recuperan y se aplica la regla de rechazo o `NULL`.
- **cantidadProductos:** los dañados tienen un carácter extra (`2x`, `10K`) y, al quitarlo, queda 1 o 2 dígitos, igual que en los válidos. Además, la distribución por cantidad (1 a 10) de los recuperados es casi idéntica a la de los válidos (unas 500 ventas por valor). Esto indica que el dato original se conserva debajo del ruido, por lo que se recupera quitando los caracteres no numéricos.
- **idArticulo:** la longitud tras limpiar (moda de 3 dígitos) coincide con la de los válidos, lo que apoya recuperarlo quitando los caracteres no numéricos. Es una regla provisional.
- **cantidadArticulo:** tras limpiar, la moda es de 3 dígitos, pero en los válidos es de 2. La diferencia indica que no se puede asegurar el valor original, así que se guarda `NULL`.
- **precioTotal:** el rango de 100 a 999,97 no tiene valores fuera de lo esperable ni inválidos, por lo que no requiere corrección.

**Reglas por campo**

Todo registro que no se pueda recuperar **no se descarta en silencio**: se guarda en `resultados/` (archivo de rechazados) con el motivo. Solo los registros validados llegan a la base relacional, protegida además por las restricciones del DDL (`PRIMARY KEY`, `FOREIGN KEY` con `PRAGMA foreign_keys = ON`, `NOT NULL` y `CHECK`).

| Campo | Problema | Regla de validación | Transformación | Si no se puede recuperar | Cómo se evita que llegue a la BD relacional |
|---|---|---|---|---|---|
| personas.numeroDocumento | 4.986 valores con símbolos o letras que reemplazan dígitos (por ejemplo `@10@93@!65`). Tras limpiar quedan 1 a 9 dígitos frente a 10 en los válidos. | Solo dígitos (`^\d+$`). | Ninguna: un documento dañado no se reconstruye, porque se inventaría otro. | La persona va a rechazados con su motivo. | Es la clave primaria: `PRIMARY KEY` y `NOT NULL`. Mojo no entrega registros con documento inválido. |
| personas.telefono | 4.969 valores dañados con el mismo patrón que el documento. | Solo dígitos. | Ninguna. | Se guarda `NULL` y la persona se conserva, porque el teléfono no es clave y el resto de sus datos es útil. | Columna que admite `NULL`, nunca un valor dañado. Mojo envía `NULL` en vez del texto corrupto. |
| articulos.idArticulo | 4.759 valores con letras mezcladas (`T474`, `cO50`); además, la clave se repite (6.013 distintos en 10.000 registros). | Solo dígitos. | Quitar los caracteres no numéricos y convertir a entero (regla provisional). Los duplicados se resuelven en la deduplicación. | Si no queda ningún dígito, el artículo va a rechazados (en los datos hay 0 casos). | `PRIMARY KEY` y tipo `INTEGER`. Ningún id duplicado o no numérico pasa del paso de Mojo. |
| articulos.cantidadArticulo | 4.703 valores con letras mezcladas (`m18`, `9rJ417`). La longitud tras limpiar no coincide con la de los válidos. | Solo dígitos. | Ninguna: no hay forma de saber cuál era el valor original. | Se guarda `NULL` y el artículo se conserva. | `CHECK` de que sea `NULL` o entero positivo. |
| ventas.idComprador | 5.039 valores dañados con el mismo patrón que el documento. | Solo dígitos. | Ninguna. | La venta va a rechazados: sin comprador identificable no puede relacionarse. | `FOREIGN KEY` hacia `personas`: si el comprador no existe, la inserción falla y no se carga. |
| ventas.idArticulo | Sin valores inválidos (todos enteros). Riesgo: es `int` en ventas y texto en artículos, y puede apuntar a artículos que no existen. | Numérico. | Unificar el tipo a `INTEGER` en ambas tablas. | Si el artículo referenciado no existe tras la limpieza, la venta va a rechazados como huérfana. | `FOREIGN KEY` hacia `articulos`. La integridad referencial se verifica en su propia sección. |
| ventas.cantidadProductos | 4.969 valores con una letra agregada (`2x`, `10K`). Al quitarla, la distribución queda uniforme entre 1 y 10, igual a la de los válidos. | Solo dígitos y valor entre 1 y 10. | Quitar los caracteres no numéricos y convertir a entero. | Si no queda un dígito, o queda fuera de 1 a 10, la venta va a rechazados. | `NOT NULL` y `CHECK (cantidadProductos BETWEEN 1 AND 10)`. |
| ventas.precioTotal | Sin valores inválidos (rango 100 a 999,97). | Numérico y mayor que 0. | Ninguna (solo conversión a número real). | La venta va a rechazados; el precio no se imputa. | `CHECK (precioTotal > 0)` y `NOT NULL`. |

# 6. Docker: diseño de la infraestructura

El equipo debe construir la infraestructura, no copiar una configuración terminada.

### Servicios mínimos

| Servicio | Responsabilidad |
|---|---|
| `mongo_personas` | documentos de PERSONAS |
| `mongo_articulos` | documentos de ARTICULOS |
| `mongo_ventas` | documentos de VENTAS |
| `relacional` | almacenamiento final relacional |

El componente Mojo puede ejecutarse como proceso del entorno de trabajo o dentro de un contenedor adicional, siempre que quede claramente documentado.

### Requisito

El `compose.yaml` o `docker-compose.yml` debe ser **generado desde el notebook** para que el único entregable sea autosuficiente.

## 6.1 Variables de infraestructura

- **Red Docker:** `red_bdm` (driver `bridge`). Una red propia aísla los 4 contenedores y les permite verse por nombre.
- **Mongo PERSONAS:** contenedor `mongo_personas`, imagen `mongo:7`, puerto `27017` del equipo → `27017` del contenedor.
- **Mongo ARTICULOS:** contenedor `mongo_articulos`, imagen `mongo:7`, puerto `27018` del equipo → `27017` del contenedor.
- **Mongo VENTAS:** contenedor `mongo_ventas`, imagen `mongo:7`, puerto `27019` del equipo → `27017` del contenedor.
- **Base relacional:** contenedor `relacional` (Alpine con cliente `sqlite3`). SQLite es un archivo, no un servidor: la base queda en `resultados/` y se comparte con el contenedor por volumen.
- **Servicio Mojo:** se ejecuta como proceso del entorno de trabajo (WSL, con `pixi run mojo`), no en contenedor.

**Justificación**

- **Puertos:** los tres Mongo escuchan en `27017` dentro del contenedor, así que en el equipo se publican en puertos distintos (`27017`, `27018`, `27019`) para que no choquen y cada nodo se alcance por separado desde el notebook.
- **Nombres:** `container_name` coincide con el nombre del servicio y con la fuente que guarda (personas, artículos, ventas), para que cualquier comando `docker` sea legible.
- **Dependencias:** los cuatro contenedores son independientes entre sí, por eso no se usa `depends_on`. Quien depende de todos es Mojo, que corre fuera: lee los tres Mongo y escribe en la base relacional. Cada Mongo tiene un `healthcheck` (ping) para que el notebook espere a que estén listos antes de cargar datos.
- **Persistencia:** cada Mongo usa un volumen con nombre, así los datos sobreviven a `docker compose down`. La carpeta `data/` se monta en solo lectura en `/datos`, desde donde se cargan los JSON sin tocar los originales.
- **Mojo fuera de Docker:** ya está instalado en WSL y el enunciado permite correrlo como proceso del entorno; evita construir una imagen adicional.

In [7]:
RED = "red_bdm"
IMG_MONGO = "mongo:7"
PUERTO_PERSONAS, PUERTO_ARTICULOS, PUERTO_VENTAS = 27017, 27018, 27019

compose_path = DOCKER_DIR / "docker-compose.yml"

compose_content = f"""# Infraestructura del proyecto BDM (generado desde el notebook)
name: bdm

services:
  mongo_personas:
    image: {IMG_MONGO}
    container_name: mongo_personas
    ports:
      - "{PUERTO_PERSONAS}:27017"
    volumes:
      - vol_personas:/data/db
      - ../data:/datos:ro
    networks:
      - {RED}
    healthcheck:
      test: ["CMD", "mongosh", "--quiet", "--eval", "db.adminCommand('ping').ok"]
      interval: 5s
      timeout: 5s
      retries: 10

  mongo_articulos:
    image: {IMG_MONGO}
    container_name: mongo_articulos
    ports:
      - "{PUERTO_ARTICULOS}:27017"
    volumes:
      - vol_articulos:/data/db
      - ../data:/datos:ro
    networks:
      - {RED}
    healthcheck:
      test: ["CMD", "mongosh", "--quiet", "--eval", "db.adminCommand('ping').ok"]
      interval: 5s
      timeout: 5s
      retries: 10

  mongo_ventas:
    image: {IMG_MONGO}
    container_name: mongo_ventas
    ports:
      - "{PUERTO_VENTAS}:27017"
    volumes:
      - vol_ventas:/data/db
      - ../data:/datos:ro
    networks:
      - {RED}
    healthcheck:
      test: ["CMD", "mongosh", "--quiet", "--eval", "db.adminCommand('ping').ok"]
      interval: 5s
      timeout: 5s
      retries: 10

  relacional:
    image: alpine:3.20
    container_name: relacional
    command: sh -c "apk add --no-cache sqlite && tail -f /dev/null"
    volumes:
      - ../resultados:/datos
    networks:
      - {RED}

networks:
  {RED}:
    name: {RED}
    driver: bridge

volumes:
  vol_personas:
  vol_articulos:
  vol_ventas:
"""

compose_path.write_text(compose_content, encoding="utf-8")
print("Ruta prevista:", compose_path)
print("Archivo generado:", compose_path.exists(), "-", compose_path.stat().st_size, "bytes")

Ruta prevista: /root/proyectoBDM/docker/docker-compose.yml
Archivo generado: True - 1483 bytes


In [8]:
r = subprocess.run(["docker", "compose", "-f", str(compose_path), "config", "--quiet"],
                   capture_output=True, text=True)
print("Código de salida:", r.returncode, "(0 = compose válido)")
print(r.stderr or "Sin errores de sintaxis")

Código de salida: 0 (0 = compose válido)
Sin errores de sintaxis


**Interpretación**

El archivo `docker/docker-compose.yml` se generó desde el notebook (1.483 bytes) y `docker compose config` terminó con código 0. Eso significa que Docker Compose lo leyó completo, resolvió las variables (red, puertos, volúmenes) y no encontró errores de sintaxis ni referencias rotas. Es una validación estática: confirma que la definición de los cuatro servicios (`mongo_personas`, `mongo_articulos`, `mongo_ventas` y `relacional`) es correcta, pero todavía no demuestra que los contenedores arranquen. Esa comprobación se hace en el paso siguiente, al levantarlos y revisar su estado.

## 6.2 Levantamiento de contenedores

Conserve evidencia de:

- `docker compose config`
- `docker compose up -d`
- `docker compose ps`

Debe demostrarse que los servicios están activos y pertenecen a la misma red.

In [9]:
import time

COMPOSE = ["docker", "compose", "-f", str(compose_path)]

def correr(cmd):
    print(f"\n$ {' '.join(cmd)}")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout, end="")
    if r.stderr.strip():
        print(r.stderr, end="")
    print(f"[código de salida: {r.returncode}]")
    return r

# 1) Validar la configuración
correr(COMPOSE + ["config"])

# 2) Levantar los contenedores (la primera vez descarga las imágenes)
r = correr(COMPOSE + ["up", "-d"])
if r.returncode != 0:
    raise RuntimeError("docker compose up -d falló; revisar el mensaje anterior")

# 3) Esperar a que los tres Mongo estén sanos y a que sqlite3 esté instalado en 'relacional'
def salud(nombre):
    r = subprocess.run(["docker", "inspect", "--format", "{{.State.Health.Status}}", nombre],
                       capture_output=True, text=True)
    return r.stdout.strip()

def sqlite_listo():
    r = subprocess.run(["docker", "exec", "relacional", "sqlite3", "--version"],
                       capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None

nodos = ["mongo_personas", "mongo_articulos", "mongo_ventas"]
inicio = time.time()
while time.time() - inicio < 120:
    estados = {n: salud(n) for n in nodos}
    version = sqlite_listo()
    if all(e == "healthy" for e in estados.values()) and version:
        break
    time.sleep(3)
print("\nEstado de salud:", estados)
print("sqlite3 en 'relacional':", version)
print(f"Tiempo de espera: {time.time() - inicio:.0f} s")

# 3b) Estado de los servicios
correr(COMPOSE + ["ps"])

# 4) Comprobar que los 4 contenedores pertenecen a la misma red
r = subprocess.run(["docker", "network", "inspect", RED], capture_output=True, text=True)
red = json.loads(r.stdout)[0]
print(f"\nRed: {red['Name']} | driver: {red['Driver']} | contenedores conectados: {len(red['Containers'])}")
for c in red["Containers"].values():
    print(f"  {c['Name']:<16} {c['IPv4Address']}")


$ docker compose -f /root/proyectoBDM/docker/docker-compose.yml config
name: bdm
services:
  mongo_articulos:
    container_name: mongo_articulos
    healthcheck:
      test:
        - CMD
        - mongosh
        - --quiet
        - --eval
        - db.adminCommand('ping').ok
      timeout: 5s
      interval: 5s
      retries: 10
    image: mongo:7
    networks:
      red_bdm: null
    ports:
      - mode: ingress
        target: 27017
        published: "27018"
        protocol: tcp
    volumes:
      - type: volume
        source: vol_articulos
        target: /data/db
        volume: {}
      - type: bind
        source: /root/proyectoBDM/data
        target: /datos
        read_only: true
        bind: {}
  mongo_personas:
    container_name: mongo_personas
    healthcheck:
      test:
        - CMD
        - mongosh
        - --quiet
        - --eval
        - db.adminCommand('ping').ok
      timeout: 5s
      interval: 5s
      retries: 10
    image: mongo:7
    networks:
    

**Interpretación de la evidencia**

- **`docker compose config` (código 0):** Compose resolvió el archivo completo: la red `red_bdm`, los tres volúmenes con nombre (`bdm_vol_*`), las carpetas montadas (`data/` en solo lectura y `resultados/`) y los puertos `27017`, `27018` y `27019` hacia el `27017` interno de cada Mongo. Valida la definición, pero no que los contenedores funcionen.
- **`docker compose up -d` (código 0):** los cuatro contenedores (`mongo_personas`, `mongo_articulos`, `mongo_ventas`, `relacional`) están en ejecución.
- **Estado de salud:** los tres Mongo respondieron `healthy`, es decir, el `healthcheck` (ping con `mongosh`) tuvo éxito y cada nodo ya acepta conexiones. `relacional` no tiene `healthcheck`; se verificó ejecutando `sqlite3 --version` dentro del contenedor (3.45.3), lo que prueba que el cliente SQLite quedó instalado.
- **`docker compose ps`:** los cuatro servicios están en `Up`. Los puertos publicados (`27017`, `27018`, `27019`) son distintos en el equipo, aunque dentro de cada contenedor todos usan `27017`, lo que confirma que no hay conflictos y que cada nodo se alcanza por separado. `relacional` no publica puertos porque SQLite es un archivo.
- **Red `red_bdm`:** los cuatro contenedores aparecen conectados, cada uno con su IP (`172.20.0.2` a `172.20.0.5`) en la misma red. Esto demuestra que pertenecen a la misma red y pueden comunicarse entre sí por nombre.

**Conclusión:** la infraestructura está lista. Hay tres nodos MongoDB independientes (uno por fuente) y un contenedor para la base relacional, todos en la misma red. Lo que sigue es cargar los JSON en cada nodo.

# 7. Carga de los JSON en MongoDB

Cada fuente debe almacenarse en su propio nodo:

- PERSONAS → `mongo_personas`
- ARTICULOS → `mongo_articulos`
- VENTAS → `mongo_ventas`

Debe demostrarse:

1. Carga.
2. Conteo.
3. Consulta de documentos.
4. Estructura.
5. Independencia de cada nodo.

**No utilice una única instancia MongoDB para simular tres servidores.**

In [10]:
# Documentación de la carga: contenedor, base de datos, colección, archivo y puerto del equipo
CARGAS = [
    ("mongo_personas",  "db_personas",  "personas",  "personas.json",  PUERTO_PERSONAS),
    ("mongo_articulos", "db_articulos", "articulos", "articulos.json", PUERTO_ARTICULOS),
    ("mongo_ventas",    "db_ventas",    "ventas",    "ventas.json",    PUERTO_VENTAS),
]

for contenedor, bd, coleccion, archivo, _ in CARGAS:
    cmd = ["docker", "exec", contenedor, "mongoimport",
           "--db", bd, "--collection", coleccion,
           "--file", f"/datos/{archivo}", "--jsonArray", "--drop"]
    print(f"\n$ {' '.join(cmd)}")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    print(f"[código de salida: {r.returncode}]")
    if r.returncode != 0:
        raise RuntimeError(f"Falló la carga en {contenedor}")


$ docker exec mongo_personas mongoimport --db db_personas --collection personas --file /datos/personas.json --jsonArray --drop
2026-10-15T23:12:03.552+0000	connected to: mongodb://localhost/
2026-10-15T23:12:03.554+0000	dropping: `db_personas.personas`
2026-10-15T23:12:04.202+0000	10000 document(s) imported successfully. 0 document(s) failed to import.
[código de salida: 0]

$ docker exec mongo_articulos mongoimport --db db_articulos --collection articulos --file /datos/articulos.json --jsonArray --drop
2026-10-15T23:12:04.447+0000	connected to: mongodb://localhost/
2026-10-15T23:12:04.448+0000	dropping: `db_articulos.articulos`
2026-10-15T23:12:04.823+0000	10000 document(s) imported successfully. 0 document(s) failed to import.
[código de salida: 0]

$ docker exec mongo_ventas mongoimport --db db_ventas --collection ventas --file /datos/ventas.json --jsonArray --drop
2026-10-15T23:12:05.020+0000	connected to: mongodb://localhost/
2026-10-15T23:12:05.023+0000	dropping: `db_ventas.vent

**Interpretación de la carga**

- Cada fuente se cargó en su propio contenedor con `mongoimport`: personas en `mongo_personas`, artículos en `mongo_articulos` y ventas en `mongo_ventas`. No se usó una única instancia para simular tres servidores.
- Los tres comandos terminaron con código de salida 0 y reportaron **10.000 documentos importados y 0 fallidos**. Eso coincide con los 10.000 registros que mostró la celda de conteo de los JSON originales, es decir, no se perdió ni se duplicó ningún registro en la carga.
- El mensaje `dropping` confirma que `--drop` vació la colección antes de importar. Gracias a eso la celda se puede ejecutar varias veces sin acumular documentos repetidos.
- La carga es fiel al original: MongoDB guarda los datos tal como vienen, incluidos los valores dañados que detectamos en 5.1. La limpieza no se hace aquí, sino en Mojo durante la transformación.
- Esta salida solo prueba que el importador terminó bien. Que los documentos estén realmente en cada nodo, con su estructura y de forma independiente, se comprueba en 7.1 consultando cada servidor por su propio puerto.

In [11]:
from pymongo import MongoClient

def conectar(puerto):
    return MongoClient("localhost", puerto, serverSelectionTimeoutMS=5000)

print("CONSULTA DE DOCUMENTOS")
for contenedor, bd, coleccion, archivo, puerto in CARGAS:
    print(f"  {bd}.{coleccion}:")
    for doc in conectar(puerto)[bd][coleccion].find({}, {"_id": 0}).limit(2):
        print("   ", doc)
v = conectar(PUERTO_VENTAS)["db_ventas"]["ventas"].find_one({"idVenta": 1}, {"_id": 0})
print("  Consulta con filtro (ventas con idVenta = 1):", v)

print("\nESTRUCTURA (campo: tipo en el primer documento)")
for contenedor, bd, coleccion, archivo, puerto in CARGAS:
    doc = conectar(puerto)[bd][coleccion].find_one({}, {"_id": 0})
    print(f"  {bd}.{coleccion}:", {k: type(x).__name__ for k, x in doc.items()})

CONSULTA DE DOCUMENTOS
  db_personas.personas:
    {'numeroDocumento': '@10@93@!65', 'nombres': 'Samuel', 'primerApellido': 'Roberts', 'segundoApellido': 'Sweeney', 'fechaNacimiento': '1938-05-30', 'telefono': '7546027921', 'direccion': '8818 Evans Cove Apt. 968, North Hannahville, WY 96152', 'email': 'bondanthony@gmail.com'}
    {'numeroDocumento': '3631787826', 'nombres': 'Mark', 'primerApellido': 'Harper', 'segundoApellido': 'White', 'fechaNacimiento': '1940-12-27', 'telefono': '6032289435', 'direccion': '047 Smith Ferry Apt. 797, Port Andrewmouth, MO 28667', 'email': 'michaelpeters@hotmail.com'}
  db_articulos.articulos:
    {'idArticulo': '791', 'nombreArticulo': 'Fractal Design Meshify', 'precioArticulo': 27.47, 'cantidadArticulo': '26'}
    {'idArticulo': '70', 'nombreArticulo': 'Bose QuietComfort Headphones', 'precioArticulo': 339.93, 'cantidadArticulo': '9rJ417'}
  db_ventas.ventas:
    {'idVenta': 1, 'idComprador': 'q61@A!213A', 'idArticulo': 2358, 'cantidadProductos': '2x', 

**Interpretación de la consulta y la estructura**

- **Consulta de documentos:** cada colección responde desde su propio nodo y devuelve los documentos con los mismos campos y valores que los JSON originales. La consulta con filtro (`idVenta = 1`) devuelve la venta esperada, lo que confirma que los datos son consultables por campo y no solo recuperables en bloque. En las consultas se excluyó el campo `_id`, que MongoDB agrega automáticamente a cada documento.
- **Los datos dañados llegaron intactos:** el primer documento de personas trae el documento `@10@93@!65`, el segundo trae el teléfono `422$9Aq9@1`, y la venta 1 trae el comprador `q61@A!213A` y la cantidad `2x`. Es el comportamiento esperado: MongoDB almacena la fuente tal como llega, y la limpieza se hace después en Mojo.
- **Estructura:** los tipos coinciden con los del perfilamiento. En personas todos los campos son texto; en artículos, `precioArticulo` es `float` y el resto texto; en ventas, `idVenta` e `idArticulo` son `int`, `precioTotal` es `float` y `idComprador` y `cantidadProductos` son texto. Esto confirma la inconsistencia ya detectada: `idArticulo` es `int` en ventas y texto en artículos, por lo que habrá que unificar el tipo antes de relacionar ambas tablas.
- **Los campos que deberían ser numéricos pero son texto** (documento, teléfono, `idComprador`, `idArticulo` de artículos, cantidades) son los mismos que traen valores dañados, lo que explica por qué MongoDB, al no imponer un esquema, aceptó todo sin validar.
- **Independencia de los nodos:** cada consulta se hizo contra un servidor distinto (puertos 27017, 27018 y 27019). La comprobación de que cada colección existe solo en su nodo se presenta en el punto 7.1.

## 7.1 Evidencia de distribución

Responda con evidencia:

- ¿Cuántos documentos quedaron en PERSONAS?
- ¿Cuántos en ARTICULOS?
- ¿Cuántos en VENTAS?
- ¿Cómo comprobó que cada colección está en su nodo correspondiente?

In [12]:
def mongosh(contenedor, js):
    r = subprocess.run(["docker", "exec", contenedor, "mongosh", "--quiet", "--eval", js],
                       capture_output=True, text=True)
    return r.stdout.strip() or r.stderr.strip()

print("1) CONTEO: cada colección consultada DENTRO de su contenedor")
for contenedor, bd, coleccion, archivo, puerto in CARGAS:
    n = mongosh(contenedor, f"db.getSiblingDB('{bd}').{coleccion}.countDocuments({{}})")
    print(f"  {contenedor:<16} {bd}.{coleccion:<10} -> {n} documentos (JSON original: {len(globals()[coleccion])})")

print("\n2) BASES DE DATOS PROPIAS de cada contenedor (sin las del sistema)")
js_bases = ("db.adminCommand({listDatabases: 1}).databases.map(d => d.name)"
            ".filter(n => !['admin', 'config', 'local'].includes(n)).join(', ')")
for contenedor, *_ in CARGAS:
    print(f"  {contenedor:<16} -> {mongosh(contenedor, js_bases)}")

print("\n3) PRUEBA CRUZADA: lo que NO debe existir en cada contenedor")
for contenedor, bd, *_ in CARGAS:
    for _, bd_otro, col_otro, _, _ in CARGAS:
        if bd_otro != bd:
            n = mongosh(contenedor, f"db.getSiblingDB('{bd_otro}').{col_otro}.countDocuments({{}})")
            print(f"  {contenedor:<16} {bd_otro}.{col_otro:<10} -> {n} documentos")

1) CONTEO: cada colección consultada DENTRO de su contenedor
  mongo_personas   db_personas.personas   -> 10000 documentos (JSON original: 10000)
  mongo_articulos  db_articulos.articulos  -> 10000 documentos (JSON original: 10000)
  mongo_ventas     db_ventas.ventas     -> 10000 documentos (JSON original: 10000)

2) BASES DE DATOS PROPIAS de cada contenedor (sin las del sistema)
  mongo_personas   -> db_personas
  mongo_articulos  -> db_articulos
  mongo_ventas     -> db_ventas

3) PRUEBA CRUZADA: lo que NO debe existir en cada contenedor
  mongo_personas   db_articulos.articulos  -> 0 documentos
  mongo_personas   db_ventas.ventas     -> 0 documentos
  mongo_articulos  db_personas.personas   -> 0 documentos
  mongo_articulos  db_ventas.ventas     -> 0 documentos
  mongo_ventas     db_personas.personas   -> 0 documentos
  mongo_ventas     db_articulos.articulos  -> 0 documentos


**Interpretación de la evidencia de distribución**

- **¿Cuántos documentos quedaron?** PERSONAS: 10.000. ARTICULOS: 10.000. VENTAS: 10.000. Cada conteo se hizo *dentro* de su contenedor (`docker exec ... mongosh`) y coincide con los 10.000 registros del JSON original, por lo que la carga no perdió ni duplicó registros.
- **¿Cómo se comprobó que cada colección está en su nodo?** De tres formas:
  1. El conteo se ejecutó dentro de cada contenedor, así que la colección existe en ese servidor y no en otro.
  2. Cada contenedor tiene su propia base (`db_personas`, `db_articulos`, `db_ventas`) con la colección de su fuente.
  3. En la prueba cruzada, las seis combinaciones que no deberían existir (por ejemplo `db_ventas.ventas` dentro de `mongo_personas`) devolvieron 0 documentos.
- **Independencia de los nodos:** son tres contenedores distintos, cada uno con su volumen y su puerto, y ninguno contiene datos de otra fuente. No se usó una única instancia para simular tres servidores.

# 8. Mojo: lenguaje y organización modular

Mojo debe utilizarse para la lógica central del procesamiento.

La CLI de Mojo permite ejecutar y compilar programas Mojo, por ejemplo con `mojo archivo.mojo`, y dispone de comandos como `mojo run` y `mojo build`.

Documentación oficial:
- https://docs.modular.com/mojo/
- https://mojolang.org/docs/cli/

### Requisito

Crear como mínimo una estructura equivalente a:

```text
mojo/
└── src/
    ├── main.mojo
    ├── limpieza.mojo
    ├── validacion.mojo
    └── etl.mojo
```

El nombre exacto puede cambiar si existe una justificación técnica.

## 8.1 Contrato funcional obligatorio

La solución debe cubrir como mínimo:

### Limpieza
`limpiar_valor_numerico`

### PERSONAS
`validar_persona`

### ARTICULOS
`validar_articulo`

### VENTAS
`validar_venta`

### Integridad
`validar_integridad_referencial`

### ETL
`ejecutar_etl`

Cada función debe documentar:

- Propósito.
- Entradas.
- Salidas.
- Regla de validación.
- Manejo de errores.
- Evidencia de ejecución.

In [14]:
(MOJO_DIR / "src").mkdir(parents=True, exist_ok=True)
(MOJO_DIR / "tests").mkdir(parents=True, exist_ok=True)

print("Estructura Mojo preparada:", MOJO_DIR)

Estructura Mojo preparada: /root/proyectoBDM/mojo


In [15]:
LIMPIEZA_MOJO = '''"""Módulo de limpieza del proyecto BDM.

Responsabilidad de Mojo: limpiar valores numéricos dañados antes de validarlos.
Regla de negocio (sección 5.1): solo se recuperan los campos donde el daño
AGREGA caracteres (cantidadProductos, idArticulo). Los identificadores donde el
daño REEMPLAZA dígitos (documento, teléfono, idComprador) no se reconstruyen.
"""


def es_solo_digitos(valor: String) -> Bool:
    """Indica si el valor es un entero sin signo escrito solo con dígitos.

    Propósito: aplicar la regla de validación estricta de los identificadores.
    Entradas: valor (String), el texto tal como llegó de MongoDB.
    Salidas: True si tiene al menos un carácter y todos son dígitos 0-9; False
        en cualquier otro caso (vacío, letras, símbolos, espacios o signo).
    Regla de validación: expresión equivalente a ^[0-9]+$.
    Manejo de errores: no lanza errores; un valor vacío devuelve False.
    Evidencia de ejecución: mojo/tests/test_limpieza.mojo.
    """
    return valor.is_ascii_digit()


def limpiar_valor_numerico(valor: String) -> String:
    """Conserva únicamente los dígitos de un valor dañado.

    Propósito: recuperar valores como "2x" o "10K" donde el ruido se agregó
        al número original.
    Entradas: valor (String), el texto tal como llegó de MongoDB.
    Salidas: String con solo los dígitos del valor, en su orden original.
        Un String vacío significa que el valor NO es recuperable.
    Regla de validación: se conservan los caracteres 0-9 y se descarta el resto.
        El llamador debe tratar el resultado vacío como dato no recuperable.
    Manejo de errores: no lanza errores. Un valor vacío o sin dígitos devuelve
        un String vacío, y el módulo de validación decide qué hacer con él
        (rechazar el registro o guardar NULL).
    Evidencia de ejecución: mojo/tests/test_limpieza.mojo.
    """
    var limpio = String()
    for c in valor.codepoint_slices():
        if String(c).is_ascii_digit():
            limpio += c
    return limpio
'''

(MOJO_DIR / "src" / "limpieza.mojo").write_text(LIMPIEZA_MOJO, encoding="utf-8")
print("Escrito:", MOJO_DIR / "src" / "limpieza.mojo")

Escrito: /root/proyectoBDM/mojo/src/limpieza.mojo


In [29]:
VALIDACION_MOJO = '''"""Módulo de validación del proyecto BDM.

Cada función recibe el registro como lista de campos (en el orden del JSON) y
devuelve el motivo de rechazo. Un String vacío significa registro válido.
Si un campo se puede limpiar, la función lo corrige en la propia lista; un
campo que pasa a NULL queda como String vacío.
Orden de campos:
    persona : numeroDocumento, nombres, primerApellido, segundoApellido,
              fechaNacimiento, telefono, direccion, email
    artículo: idArticulo, nombreArticulo, precioArticulo, cantidadArticulo
    venta   : idVenta, idComprador, idArticulo, cantidadProductos, precioTotal
"""

from std.collections import Set
from limpieza import es_solo_digitos, limpiar_valor_numerico


def es_numero_positivo(valor: String) -> Bool:
    """Indica si el valor es un número decimal mayor que cero.

    Propósito: validar precios sin depender de la conversión a decimal.
    Entradas: valor (String).
    Salidas: True si solo contiene dígitos y a lo sumo un punto, con al menos
        un dígito distinto de cero; False en cualquier otro caso.
    Regla de validación: solo dígitos y a lo sumo un punto decimal, con valor > 0.
    Manejo de errores: no lanza errores; vacío o con símbolos devuelve False.
    Evidencia de ejecución: mojo/tests/test_validacion.mojo.
    """
    var hay_digito = False
    var hay_distinto_de_cero = False
    var puntos = 0
    for c in valor.codepoint_slices():
        var s = String(c)
        if s.is_ascii_digit():
            hay_digito = True
            if s != "0":
                hay_distinto_de_cero = True
        elif s == ".":
            puntos += 1
            if puntos > 1:
                return False
        else:
            return False
    return hay_digito and hay_distinto_de_cero


def validar_persona(mut campos: List[String]) -> String:
    """Valida un registro de PERSONAS.

    Propósito: decidir si la persona puede cargarse a la base relacional.
    Entradas: campos (8 valores); se modifica en el lugar.
    Salidas: "" si es válida; si no, el motivo de rechazo.
    Regla de validación: numeroDocumento (clave primaria) debe ser solo dígitos
        y no se reconstruye; telefono dañado pasa a NULL y la persona se conserva.
    Manejo de errores: no lanza errores; una lista con otro número de campos
        se rechaza por estructura.
    Evidencia de ejecución: mojo/tests/test_validacion.mojo.
    """
    if len(campos) != 8:
        return "estructura: se esperaban 8 campos"
    if not es_solo_digitos(campos[0]):
        return "numeroDocumento inválido"
    if not es_solo_digitos(campos[5]):
        campos[5] = String("")
    return ""


def validar_articulo(mut campos: List[String]) -> String:
    """Valida un registro de ARTICULOS.

    Propósito: decidir si el artículo puede cargarse a la base relacional.
    Entradas: campos (4 valores); se modifica en el lugar.
    Salidas: "" si es válido; si no, el motivo de rechazo.
    Regla de validación: idArticulo se recupera quitando los caracteres no
        numéricos y se rechaza si no queda ningún dígito; precioArticulo debe
        ser un número mayor que cero; cantidadArticulo dañada pasa a NULL.
    Manejo de errores: no lanza errores; otro número de campos se rechaza por
        estructura.
    Evidencia de ejecución: mojo/tests/test_validacion.mojo.
    """
    if len(campos) != 4:
        return "estructura: se esperaban 4 campos"
    var id_limpio = limpiar_valor_numerico(campos[0])
    if id_limpio.byte_length() == 0:
        return "idArticulo no recuperable"
    campos[0] = id_limpio
    if not es_numero_positivo(campos[2]):
        return "precioArticulo inválido"
    if not es_solo_digitos(campos[3]):
        campos[3] = String("")
    return ""


def validar_venta(mut campos: List[String]) -> String:
    """Valida un registro de VENTAS.

    Propósito: decidir si la venta puede cargarse a la base relacional.
    Entradas: campos (5 valores); se modifica en el lugar.
    Salidas: "" si es válida; si no, el motivo de rechazo.
    Regla de validación: idVenta, idComprador e idArticulo deben ser solo
        dígitos (idComprador es llave foránea y no se reconstruye);
        cantidadProductos se recupera quitando caracteres no numéricos y debe
        quedar entre 1 y 10; precioTotal debe ser un número mayor que cero.
    Manejo de errores: no lanza errores; otro número de campos se rechaza por
        estructura.
    Evidencia de ejecución: mojo/tests/test_validacion.mojo.
    """
    if len(campos) != 5:
        return "estructura: se esperaban 5 campos"
    if not es_solo_digitos(campos[0]):
        return "idVenta inválido"
    if not es_solo_digitos(campos[1]):
        return "idComprador inválido"
    if not es_solo_digitos(campos[2]):
        return "idArticulo inválido"
    var cantidad = limpiar_valor_numerico(campos[3])
    var en_rango = (cantidad.byte_length() == 1 and cantidad != "0") or cantidad == "10"
    if not en_rango:
        return "cantidadProductos no recuperable o fuera de rango 1-10"
    campos[3] = cantidad
    if not es_numero_positivo(campos[4]):
        return "precioTotal inválido"
    return ""
    
def validar_integridad_referencial(
    id_comprador: String,
    id_articulo: String,
    personas: Set[String],
    articulos: Set[String],
) -> String:
    """Verifica que una venta apunte a una persona y a un artículo existentes.

    Propósito: impedir que lleguen a la base relacional ventas huérfanas, es
        decir, con llaves foráneas que no tienen registro asociado.
    Entradas: id_comprador e id_articulo de la venta (ya validados como solo
        dígitos); personas y articulos, conjuntos con los ids que SÍ se cargaron.
    Salidas: "" si ambas referencias existen; si no, el motivo de rechazo.
    Regla de validación: id_comprador debe estar en personas e id_articulo en
        articulos. Se comprueba primero el comprador.
    Manejo de errores: no lanza errores; un id vacío o desconocido se rechaza
        con su motivo y la venta se guarda en rechazados, no se elimina en silencio.
    Evidencia de ejecución: mojo/tests/test_integridad.mojo.
    """
    if not (id_comprador in personas):
        return "idComprador sin persona asociada"
    if not (id_articulo in articulos):
        return "idArticulo sin artículo asociado"
    return ""
'''

(MOJO_DIR / "src" / "validacion.mojo").write_text(VALIDACION_MOJO, encoding="utf-8")
print("Escrito:", MOJO_DIR / "src" / "validacion.mojo")

Escrito: /root/proyectoBDM/mojo/src/validacion.mojo


In [32]:
ETL_MOJO = r'''"""Módulo ETL del proyecto BDM.

Responsabilidad de Mojo: leer los registros extraídos de MongoDB (archivos de
texto con columnas separadas por tabulador), validarlos, limpiarlos, comprobar
la integridad referencial y escribir los resultados.
Responsabilidad de Python (fuera de este módulo): extraer de MongoDB, escribir
los archivos de entrada y cargar los archivos limpios a SQLite.
"""

from std.collections import Set
from validacion import validar_persona, validar_articulo, validar_venta, validar_integridad_referencial


def unir(campos: List[String]) -> String:
    """Une los campos de un registro separándolos con tabulador."""
    var texto = String()
    for i in range(len(campos)):
        if i > 0:
            texto += "\t"
        texto += campos[i]
    return texto


def leer_registros(ruta: String) raises -> List[List[String]]:
    """Lee un archivo de texto y devuelve un registro por línea no vacía.

    Entradas: ruta del archivo (columnas separadas por tabulador).
    Salidas: lista de registros, cada uno como lista de campos.
    Manejo de errores: lanza un error si el archivo no existe o no se puede leer.
    """
    var archivo = open(ruta, "r")
    var contenido = archivo.read()
    archivo.close()
    var registros = List[List[String]]()
    for linea in contenido.split("\n"):
        if linea.byte_length() == 0:
            continue
        var campos = List[String]()
        for parte in linea.split("\t"):
            campos.append(String(parte))
        registros.append(campos^)
    return registros^


def escribir_texto(ruta: String, texto: String) raises:
    """Escribe el texto en el archivo, reemplazando su contenido anterior."""
    var archivo = open(ruta, "w")
    archivo.write(texto)
    archivo.close()


def ejecutar_etl(carpeta_entrada: String, carpeta_salida: String) raises -> String:
    """Ejecuta el ETL completo sobre las tres fuentes.

    Propósito: producir los datos limpios que se cargarán a la base relacional,
        sin eliminar registros en silencio.
    Entradas: carpeta_entrada con personas.tsv, articulos.tsv y ventas.tsv;
        carpeta_salida (debe existir) donde se escriben los resultados.
    Salidas: escribe personas_limpias.tsv, articulos_limpios.tsv,
        ventas_limpias.tsv, rechazados.tsv (fuente, línea, motivo, registro
        original) y resumen.txt; devuelve el texto del resumen.
    Regla de validación: validar_persona, validar_articulo y validar_venta
        sobre cada registro; idArticulo repetido se rechaza conservando la
        primera aparición; cada venta se acepta solo si su comprador y su
        artículo se cargaron (validar_integridad_referencial).
    Manejo de errores: un registro inválido no detiene el proceso: se registra
        en rechazados.tsv con su motivo. Un archivo de entrada ausente lanza error.
    Evidencia de ejecución: mojo/tests/test_etl.mojo y mojo/src/main.mojo.
    """
    var personas = leer_registros(carpeta_entrada + "/personas.tsv")
    var articulos = leer_registros(carpeta_entrada + "/articulos.tsv")
    var ventas = leer_registros(carpeta_entrada + "/ventas.tsv")

    var rechazados = String()
    var personas_ok = Set[String]()
    var articulos_ok = Set[String]()
    var salida_personas = String()
    var salida_articulos = String()
    var salida_ventas = String()
    var rech_personas = 0
    var rech_articulos = 0
    var rech_ventas = 0

    for i in range(len(personas)):
        var original = unir(personas[i])
        var motivo = validar_persona(personas[i])
        if motivo.byte_length() > 0:
            rechazados += String("personas\t", i + 1, "\t", motivo, "\t", original, "\n")
            rech_personas += 1
        else:
            var clave = personas[i][0]
            personas_ok.add(clave)
            salida_personas += unir(personas[i]) + "\n"

    for i in range(len(articulos)):
        var original = unir(articulos[i])
        var motivo = validar_articulo(articulos[i])
        if motivo.byte_length() == 0:
            var clave = articulos[i][0]
            if clave in articulos_ok:
                motivo = String("idArticulo duplicado (se conserva la primera aparición)")
        if motivo.byte_length() > 0:
            rechazados += String("articulos\t", i + 1, "\t", motivo, "\t", original, "\n")
            rech_articulos += 1
        else:
            var clave = articulos[i][0]
            articulos_ok.add(clave)
            salida_articulos += unir(articulos[i]) + "\n"

    for i in range(len(ventas)):
        var original = unir(ventas[i])
        var motivo = validar_venta(ventas[i])
        if motivo.byte_length() == 0:
            motivo = validar_integridad_referencial(ventas[i][1], ventas[i][2], personas_ok, articulos_ok)
        if motivo.byte_length() > 0:
            rechazados += String("ventas\t", i + 1, "\t", motivo, "\t", original, "\n")
            rech_ventas += 1
        else:
            salida_ventas += unir(ventas[i]) + "\n"

    var resumen = String(
        "personas: leidas=", len(personas), " validas=", len(personas) - rech_personas,
        " rechazadas=", rech_personas, "\n",
        "articulos: leidas=", len(articulos), " validos=", len(articulos) - rech_articulos,
        " rechazados=", rech_articulos, "\n",
        "ventas: leidas=", len(ventas), " validas=", len(ventas) - rech_ventas,
        " rechazadas=", rech_ventas,
    )

    escribir_texto(carpeta_salida + "/personas_limpias.tsv", salida_personas)
    escribir_texto(carpeta_salida + "/articulos_limpios.tsv", salida_articulos)
    escribir_texto(carpeta_salida + "/ventas_limpias.tsv", salida_ventas)
    escribir_texto(carpeta_salida + "/rechazados.tsv", rechazados)
    escribir_texto(carpeta_salida + "/resumen.txt", resumen)
    return resumen
'''

MAIN_MOJO = r'''"""Punto de entrada del ETL. Se ejecuta desde la raíz del proyecto, con rutas relativas."""

from etl import ejecutar_etl


def main() raises:
    var resumen = ejecutar_etl("resultados/extraccion", "resultados/limpio")
    print(resumen)
'''

(MOJO_DIR / "src" / "etl.mojo").write_text(ETL_MOJO, encoding="utf-8")
(MOJO_DIR / "src" / "main.mojo").write_text(MAIN_MOJO, encoding="utf-8")
print("Escritos:", MOJO_DIR / "src" / "etl.mojo", "y", MOJO_DIR / "src" / "main.mojo")

Escritos: /root/proyectoBDM/mojo/src/etl.mojo y /root/proyectoBDM/mojo/src/main.mojo


## 8.2 Pruebas unitarias de Mojo

Cada función central debe tener al menos tres escenarios:

- Dato válido.
- Dato inválido.
- Dato vacío/nulo o no recuperable.

El equipo debe demostrar que las funciones se ejecutan realmente con Mojo y no solamente están escritas.

In [16]:
TEST_LIMPIEZA = '''from std.testing import assert_equal, assert_true, assert_false
from limpieza import es_solo_digitos, limpiar_valor_numerico


def main() raises:
    # es_solo_digitos: válido, inválido, vacío
    assert_true(es_solo_digitos("190252420"))
    assert_false(es_solo_digitos("@10@93@!65"))
    assert_false(es_solo_digitos(""))

    # limpiar_valor_numerico: válido, inválido recuperable, no recuperable
    assert_equal(limpiar_valor_numerico("123"), String("123"))
    assert_equal(limpiar_valor_numerico("2x"), String("2"))
    assert_equal(limpiar_valor_numerico("10K"), String("10"))
    assert_equal(limpiar_valor_numerico(""), String(""))
    assert_equal(limpiar_valor_numerico("@!$"), String(""))

    print("test_limpieza: 8 comprobaciones OK")
'''

test_path = MOJO_DIR / "tests" / "test_limpieza.mojo"
test_path.write_text(TEST_LIMPIEZA, encoding="utf-8")

r = subprocess.run(["pixi", "run", "mojo", "-I", str(MOJO_DIR / "src"), str(test_path)],
                   cwd=BASE, capture_output=True, text=True)
print("Código de salida:", r.returncode)
print(r.stdout, end="")
print(r.stderr, end="")

Código de salida: 0
test_limpieza: 8 comprobaciones OK


In [30]:
TEST_VALIDACION = '''from std.testing import assert_equal, assert_true, assert_false
from validacion import validar_persona, validar_articulo, validar_venta, es_numero_positivo


def lista(*valores: String) -> List[String]:
    """Construye una List[String] a partir de los valores recibidos."""
    var resultado = List[String]()
    for i in range(len(valores)):
        resultado.append(valores[i])
    return resultado^


def main() raises:
    # es_numero_positivo: válido, inválido, vacío
    assert_true(es_numero_positivo("301.26"))
    assert_false(es_numero_positivo("0"))
    assert_false(es_numero_positivo(""))

    # validar_persona
    var p1 = lista("190252420", "Aaron", "Palmer", "Khan", "2016-05-24", "7546027921", "Calle 1", "a@b.com")
    assert_equal(validar_persona(p1), String(""))                       # válida
    assert_equal(p1[5], String("7546027921"))
    var p2 = lista("190252420", "Aaron", "Palmer", "Khan", "2016-05-24", "422$9Aq9@1", "Calle 1", "a@b.com")
    assert_equal(validar_persona(p2), String(""))                       # teléfono dañado -> NULL
    assert_equal(p2[5], String(""))
    var p3 = lista("@10@93@!65", "Samuel", "Roberts", "Sweeney", "1938-05-30", "7546027921", "Calle 1", "a@b.com")
    assert_equal(validar_persona(p3), String("numeroDocumento inválido"))   # no recuperable
    var p4 = List[String]()
    assert_equal(validar_persona(p4), String("estructura: se esperaban 8 campos"))   # vacío

    # validar_articulo
    var a1 = lista("358", "Intel i9", "301.26", "35")
    assert_equal(validar_articulo(a1), String(""))                      # válido
    assert_equal(a1[0], String("358"))
    var a2 = lista("T474", "SeaSonic Focus Plus", "237.64", "m18")
    assert_equal(validar_articulo(a2), String(""))                      # id recuperado, cantidad -> NULL
    assert_equal(a2[0], String("474"))
    assert_equal(a2[3], String(""))
    var a3 = lista("@@@", "X", "10.5", "3")
    assert_equal(validar_articulo(a3), String("idArticulo no recuperable"))
    var a4 = lista("358", "X", "0", "3")
    assert_equal(validar_articulo(a4), String("precioArticulo inválido"))
    var a5 = List[String]()
    assert_equal(validar_articulo(a5), String("estructura: se esperaban 4 campos"))   # vacío

    # validar_venta
    var v1 = lista("2", "731794571", "847", "4", "877.25")
    assert_equal(validar_venta(v1), String(""))                         # válida
    var v2 = lista("1", "731794571", "2358", "2x", "202.14")
    assert_equal(validar_venta(v2), String(""))                         # cantidad recuperada
    assert_equal(v2[3], String("2"))
    var v3 = lista("1", "731794571", "2358", "10K", "202.14")
    assert_equal(validar_venta(v3), String(""))
    assert_equal(v3[3], String("10"))
    var v4 = lista("1", "q61@A!213A", "2358", "2", "202.14")
    assert_equal(validar_venta(v4), String("idComprador inválido"))     # no recuperable
    var v5 = lista("1", "731794571", "2358", "@@", "202.14")
    assert_equal(validar_venta(v5), String("cantidadProductos no recuperable o fuera de rango 1-10"))
    var v6 = lista("1", "731794571", "2358", "99", "202.14")
    assert_equal(validar_venta(v6), String("cantidadProductos no recuperable o fuera de rango 1-10"))
    var v7 = List[String]()
    assert_equal(validar_venta(v7), String("estructura: se esperaban 5 campos"))   # vacío

    print("test_validacion: todos los escenarios OK")
'''

test_path = MOJO_DIR / "tests" / "test_validacion.mojo"
test_path.write_text(TEST_VALIDACION, encoding="utf-8")

r = subprocess.run(["pixi", "run", "mojo", "-I", str(MOJO_DIR / "src"), str(test_path)],
                   cwd=BASE, capture_output=True, text=True)
print("Código de salida:", r.returncode)
print(r.stdout, end="")
print(r.stderr, end="")

Código de salida: 0
test_validacion: todos los escenarios OK


In [31]:
TEST_INTEGRIDAD = '''from std.testing import assert_equal
from std.collections import Set
from validacion import validar_integridad_referencial


def main() raises:
    var personas = Set[String]()
    personas.add("731794571")
    personas.add("190252420")
    var articulos = Set[String]()
    articulos.add("358")
    articulos.add("847")

    # válido: comprador y artículo existen
    assert_equal(validar_integridad_referencial("731794571", "847", personas, articulos), String(""))

    # inválido: comprador inexistente / artículo inexistente
    assert_equal(
        validar_integridad_referencial("999999999", "847", personas, articulos),
        String("idComprador sin persona asociada"),
    )
    assert_equal(
        validar_integridad_referencial("731794571", "2358", personas, articulos),
        String("idArticulo sin artículo asociado"),
    )

    # vacío: id vacío y conjuntos vacíos
    assert_equal(
        validar_integridad_referencial("", "847", personas, articulos),
        String("idComprador sin persona asociada"),
    )
    var sin_personas = Set[String]()
    var sin_articulos = Set[String]()
    assert_equal(
        validar_integridad_referencial("731794571", "847", sin_personas, sin_articulos),
        String("idComprador sin persona asociada"),
    )

    print("test_integridad: todos los escenarios OK")
'''

test_path = MOJO_DIR / "tests" / "test_integridad.mojo"
test_path.write_text(TEST_INTEGRIDAD, encoding="utf-8")

r = subprocess.run(["pixi", "run", "mojo", "-I", str(MOJO_DIR / "src"), str(test_path)],
                   cwd=BASE, capture_output=True, text=True)
print("Código de salida:", r.returncode)
print(r.stdout, end="")
print(r.stderr, end="")

Código de salida: 0
test_integridad: todos los escenarios OK


In [34]:
base = RESULTADOS / "prueba_etl"
entrada, vacia = base / "entrada", base / "entrada_vacia"
for d in (entrada, vacia, base / "salida", base / "salida_vacia"):
    d.mkdir(parents=True, exist_ok=True)

def tsv(filas):
    return "".join("\t".join(f) + "\n" for f in filas)

(entrada / "personas.tsv").write_text(tsv([
    ["190252420", "Aaron", "Palmer", "Khan", "2016-05-24", "7546027921", "Calle 1", "a@b.com"],
    ["@10@93@!65", "Samuel", "Roberts", "Sweeney", "1938-05-30", "7546027921", "Calle 2", "b@b.com"],
    ["731794571", "Luz", "Gomez", "Ruiz", "1990-01-01", "422$9Aq9@1", "Calle 3", "c@b.com"],
]), encoding="utf-8")
(entrada / "articulos.tsv").write_text(tsv([
    ["358", "Intel i9", "301.26", "35"],
    ["T358", "Intel i9 repetido", "10.5", "7"],
    ["847", "Kingston", "280.17", "m18"],
    ["@@@", "X", "10.5", "3"],
]), encoding="utf-8")
(entrada / "ventas.tsv").write_text(tsv([
    ["1", "731794571", "847", "2x", "202.14"],
    ["2", "190252420", "358", "4", "877.25"],
    ["3", "999999999", "358", "1", "100.5"],
    ["4", "q61@A!213A", "358", "2", "100.5"],
    ["5", "731794571", "2358", "3", "50.0"],
]), encoding="utf-8")
for nombre in ("personas.tsv", "articulos.tsv", "ventas.tsv"):
    (vacia / nombre).write_text("", encoding="utf-8")

TEST_ETL = r'''from std.testing import assert_equal
from etl import ejecutar_etl


def leer(ruta: String) raises -> String:
    var archivo = open(ruta, "r")
    var contenido = archivo.read()
    archivo.close()
    return contenido


def main() raises:
    var base = "resultados/prueba_etl"

    # Escenario 1: datos con registros válidos, inválidos y duplicados
    var resumen = ejecutar_etl(base + "/entrada", base + "/salida")
    assert_equal(
        resumen,
        String("personas: leidas=3 validas=2 rechazadas=1\narticulos: leidas=4 validos=2 rechazados=2\nventas: leidas=5 validas=2 rechazadas=3"),
    )
    assert_equal(
        leer(base + "/salida/personas_limpias.tsv"),
        String("190252420\tAaron\tPalmer\tKhan\t2016-05-24\t7546027921\tCalle 1\ta@b.com\n731794571\tLuz\tGomez\tRuiz\t1990-01-01\t\tCalle 3\tc@b.com\n"),
    )
    assert_equal(
        leer(base + "/salida/articulos_limpios.tsv"),
        String("358\tIntel i9\t301.26\t35\n847\tKingston\t280.17\t\n"),
    )
    assert_equal(
        leer(base + "/salida/ventas_limpias.tsv"),
        String("1\t731794571\t847\t2\t202.14\n2\t190252420\t358\t4\t877.25\n"),
    )

    # Escenario 2: archivos de entrada vacíos
    var resumen_vacio = ejecutar_etl(base + "/entrada_vacia", base + "/salida_vacia")
    assert_equal(
        resumen_vacio,
        String("personas: leidas=0 validas=0 rechazadas=0\narticulos: leidas=0 validos=0 rechazados=0\nventas: leidas=0 validas=0 rechazadas=0"),
    )

    print("test_etl: todos los escenarios OK")
'''

test_path = MOJO_DIR / "tests" / "test_etl.mojo"
test_path.write_text(TEST_ETL, encoding="utf-8")

r = subprocess.run(["pixi", "run", "mojo", "-I", str(MOJO_DIR / "src"), str(test_path)],
                   cwd=BASE, capture_output=True, text=True)
print("Código de salida:", r.returncode)
print(r.stdout, end="")
print(r.stderr, end="")
print("\nRechazados del escenario 1:")
print((base / "salida" / "rechazados.tsv").read_text(encoding="utf-8"))

Código de salida: 0
test_etl: todos los escenarios OK

Rechazados del escenario 1:
personas	2	numeroDocumento inválido	@10@93@!65	Samuel	Roberts	Sweeney	1938-05-30	7546027921	Calle 2	b@b.com
articulos	2	idArticulo duplicado (se conserva la primera aparición)	T358	Intel i9 repetido	10.5	7
articulos	4	idArticulo no recuperable	@@@	X	10.5	3
ventas	3	idComprador sin persona asociada	3	999999999	358	1	100.5
ventas	4	idComprador inválido	4	q61@A!213A	358	2	100.5
ventas	5	idArticulo sin artículo asociado	5	731794571	2358	3	50.0



# 9. ETL

Implemente:

```text
EXTRACT
   ↓
PROFILE
   ↓
VALIDATE
   ↓
TRANSFORM
   ↓
REFERENTIAL CHECK
   ↓
LOAD
   ↓
VERIFY
```

### Extract
Obtener datos de los tres nodos MongoDB.

### Transform
Aplicar las reglas definidas en Mojo.

### Load
Insertar solamente registros que cumplan el modelo relacional.

### Verify
Comprobar cantidades, tipos, claves y relaciones.

## 9.1 Reglas de transformación

| Fuente | Campo | Regla |
|---|---|---|
| PERSONAS | numeroDocumento | TODO |
| PERSONAS | telefono | TODO |
| ARTICULOS | idArticulo | TODO |
| ARTICULOS | cantidadArticulo | TODO |
| VENTAS | idComprador | TODO |
| VENTAS | idArticulo | TODO |
| VENTAS | cantidadProductos | TODO |
| VENTAS | precioTotal | TODO |

### Pregunta crítica

¿Qué hace su ETL cuando un valor no puede transformarse de forma confiable?

Explique y justifique.

# 10. Base de datos relacional

Construya la base relacional dentro del notebook.

Se recomienda SQLite para facilitar la reproducibilidad.

### Requisitos

- PK en PERSONAS.
- PK en ARTICULOS.
- PK en VENTAS.
- FK `VENTAS.idComprador`.
- FK `VENTAS.idArticulo`.
- Tipos compatibles.
- Integridad referencial activada.

**El notebook debe crear la base desde cero.**

In [15]:
db_path = RESULTADOS / "almacen.sqlite"

conexion = sqlite3.connect(db_path)
conexion.execute("PRAGMA foreign_keys = ON")

# TODO: crear las tablas mediante DDL escrito por el equipo.
# No utilizar una base de datos ya poblada.

conexion.close()

print("Base relacional:", db_path)

Base relacional: /root/proyectoBDM/resultados/almacen.sqlite


# 11. Carga final y verificación

Después del ETL:

1. Consultar PERSONAS.
2. Consultar ARTICULOS.
3. Consultar VENTAS.
4. Contar registros.
5. Identificar duplicados.
6. Comprobar FK.
7. Comprobar registros huérfanos.
8. Comparar con el perfilamiento inicial.

La pérdida de registros debe poder explicarse.

In [16]:
# TODO: consultar conteos por tabla y mostrar muestras.

# conexion = sqlite3.connect(db_path)
# conexion.execute("PRAGMA foreign_keys = ON")
# ...

# 12. Pruebas de integridad referencial

Realice como mínimo:

### Prueba A
Ventas cuyo comprador no existe.

### Prueba B
Ventas cuyo artículo no existe.

### Prueba C
Duplicidad de claves.

### Prueba D
Valores nulos en campos obligatorios.

### Prueba E
Tipos incompatibles.

Cada prueba debe presentar:

**consulta → resultado → interpretación**

In [17]:
# TODO: implementar las consultas de integridad.

# 13. Consulta analítica 1 – Cinco artículos más vendidos

Construya una consulta SQL que determine los cinco artículos con mayor cantidad de unidades vendidas.

Debe mostrar:

- Identificador.
- Nombre.
- Cantidad total vendida.

Luego construya una visualización apropiada e interprete el resultado.

In [18]:
import pandas as pd
import matplotlib.pyplot as plt

consulta = """
-- TODO: escribir la consulta SQL
"""

# df = pd.read_sql_query(consulta, conexion)
# display(df)
# TODO: gráfico

# 14. Consulta analítica 2 – Cinco compradores con más compras

Determine los cinco compradores con mayor número de compras.

Diferencie claramente:

- número de transacciones
- cantidad de productos

Justifique la métrica utilizada.

In [19]:
consulta = """
-- TODO
"""

# TODO: ejecutar y graficar

# 15. Consulta analítica 3 – Distribución de precios

Obtenga los cinco primeros artículos según el criterio indicado en el proyecto y analice su distribución de precios.

El gráfico debe tener título, ejes identificados y unidades cuando correspondan.

In [20]:
consulta = """
-- TODO
"""

# TODO: ejecutar y visualizar

# 16. Consulta analítica 4 – Cinco artículos menos vendidos

Determine los cinco artículos con menor cantidad de unidades vendidas.

### Importante

Defina cómo tratará los artículos que nunca aparecen en VENTAS.

Explique por qué su consulta sí o no los incluye.

In [21]:
consulta = """
-- TODO
"""

# TODO: ejecutar y visualizar

# 17. Reproducibilidad

Demuestre que el notebook puede reconstruir la solución.

### Prueba

1. Detener los contenedores.
2. Eliminar los contenedores.
3. Eliminar la base relacional generada.
4. Ejecutar nuevamente las celdas necesarias.
5. Comprobar que los resultados se reconstruyen.

No se aceptan rutas personales ni archivos ocultos del equipo.

In [22]:
print("Python:", sys.version.split()[0])

# TODO: comprobar versiones disponibles.
# subprocess.run(["docker", "--version"], check=False)
# subprocess.run(["docker", "compose", "version"], check=False)
# subprocess.run(["mojo", "--version"], check=False)

Python: 3.12.3


# 18. Defensa técnica

Cada integrante debe estar preparado para responder individualmente:

1. ¿Por qué utilizar tres nodos MongoDB?
2. ¿Qué problema de calidad encontró?
3. ¿Qué regla utilizó para solucionarlo?
4. ¿Qué función Mojo realiza la transformación?
5. ¿Cómo se garantiza la integridad referencial?
6. ¿Qué ocurre si una venta apunta a un artículo inexistente?
7. ¿Qué parte del proyecto realiza Python?
8. ¿Qué parte realiza Mojo?
9. ¿Cómo se reconstruye la solución desde cero?
10. ¿Qué decisión técnica cambiaría y por qué?

# 19. Registro de uso de IA

| Herramienta | Uso | Aporte generado | Modificación realizada | Validación |
|---|---|---|---|---|
| TODO | TODO | TODO | TODO | TODO |

### Declaración

El equipo declara que puede explicar y defender el código presentado y que las herramientas de IA, si fueron utilizadas, fueron empleadas como apoyo y no como sustituto del aprendizaje.

# 20. Conclusiones

Escriba mínimo tres conclusiones técnicas.

Deben responder:

1. ¿Qué aprendió sobre arquitectura distribuida?
2. ¿Qué aprendió sobre calidad y ETL?
3. ¿Qué aprendió sobre Mojo?
4. ¿Qué ventajas y limitaciones encontró al integrar MongoDB, Docker, Mojo y SQLite?
5. ¿Qué mejoraría en una segunda versión?

Evite conclusiones genéricas.

# 21. Referencias

Como mínimo consulte documentación oficial:

- Mojo / Modular: https://docs.modular.com/mojo/
- Mojo CLI: https://mojolang.org/docs/cli/
- Docker Documentation: https://docs.docker.com/
- MongoDB Documentation: https://www.mongodb.com/docs/
- SQLite Documentation: https://www.sqlite.org/docs.html

Agregue las fuentes adicionales utilizadas y cite dentro del notebook cuando corresponda.

# 22. Lista de comprobación antes de entregar

- [ ] El notebook tiene los integrantes.
- [ ] No existen rutas absolutas personales.
- [ ] Los tres JSON originales no fueron modificados.
- [ ] Existen tres nodos MongoDB independientes.
- [ ] Docker levanta correctamente la arquitectura.
- [ ] Cada MongoDB contiene su fuente correspondiente.
- [ ] Se realizó perfilamiento de datos.
- [ ] Se documentaron problemas de calidad.
- [ ] Las funciones centrales están implementadas en Mojo.
- [ ] Existen pruebas de las funciones Mojo.
- [ ] El ETL está documentado.
- [ ] Se creó la BD relacional desde cero.
- [ ] Se validaron PK y FK.
- [ ] Se comprobaron registros huérfanos.
- [ ] Se realizaron las cuatro consultas analíticas.
- [ ] Se generaron los gráficos.
- [ ] Cada resultado tiene interpretación.
- [ ] Se registró el uso de IA.
- [ ] El notebook se ejecuta de principio a fin.
- [ ] Las conclusiones son propias del equipo.

# ENTREGABLE ÚNICO

## `ApellidoNombre_ProyectoBDM.ipynb`

El notebook debe contener **código + resultados + evidencias + análisis + conclusiones**.

> **No entregue ZIP. No entregue archivos adicionales. El notebook debe generar los archivos de configuración necesarios para reproducir la solución.**

### Criterio central

> **No se evalúa solamente que la solución funcione; se evalúa que el equipo pueda demostrar, explicar y defender cómo y por qué funciona.**